<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230821_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, LeakyReLU, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import Add
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, LayerNormalization, LeakyReLU, Flatten, Add, Bidirectional
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Attention

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Bidirectional, LayerNormalization


In [19]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [11]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2", "A_M1_B1_W1_Z_L", "A_M1_B1_BC_Z_L", "A_M1_B1_W2_Z_L", "A_M1_B1_W1_Z_R", "A_M1_B1_BC_Z_R", "A_M1_B1_W2_Z_R","Gauge"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # Save column names before transformation
    columns = X.columns.tolist()

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, columns

def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y, column_names = prepare_data(s30_train)

# Model building
N_TIMESTEPS = 10
N_FEATURES = 29

input_layer = Input(shape=(N_TIMESTEPS, N_FEATURES))
x = Bidirectional(LSTM(100, return_sequences=True))(input_layer)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(100, return_sequences=True))(x)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

query = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
key = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
value = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
attention_result = Attention(use_scale=True)([query, key, value])
combined = Add()([x, attention_result])


# Flatten combined outputs for Dense Layer
x = Flatten()(combined)
x = Dense(100, kernel_regularizer=l2(0.01))(x)
x = LeakyReLU()(x)
x = Dropout(0.5)(x)
output = Dense(4)(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

X_s, y_s = reshape_data(data_s_X, data_s_y)

# Splitting training/testing data
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Model training
model.fit(X_train_s, y_train_s, epochs=100, batch_size=32, validation_data=(X_test_s, y_test_s), callbacks=[es])

# Predictions
predictions_s = model.predict(X_test_s)

# Print predictions
print("Predictions for data_s")
print(predictions_s[:5])

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 10, 29)]     0           []                               
                                                                                                  
 bidirectional (Bidirectional)  (None, 10, 200)      104000      ['input_1[0][0]']                
                                                                                                  
 layer_normalization (LayerNorm  (None, 10, 200)     400         ['bidirectional[0][0]']          
 alization)                                                                                       
                                                                                                  
 leaky_re_lu (LeakyReLU)        (None, 10, 200)      0           ['layer_normalization[0][0]']

In [12]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])
# CSV 파일로 저장
predictions_30s.to_csv('/content/drive/My Drive/철도/30s_20230821_lstm.csv', index=False)

In [13]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2", "A_M1_B1_W1_Z_L", "A_M1_B1_BC_Z_L", "A_M1_B1_W2_Z_L", "A_M1_B1_W1_Z_R", "A_M1_B1_BC_Z_R", "A_M1_B1_W2_Z_R","Gauge"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # Save column names before transformation
    columns = X.columns.tolist()

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, columns

def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y, column_names = prepare_data(s40_train)

# Model building
N_TIMESTEPS = 10
N_FEATURES = 29

input_layer = Input(shape=(N_TIMESTEPS, N_FEATURES))
x = Bidirectional(LSTM(100, return_sequences=True))(input_layer)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(100, return_sequences=True))(x)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

query = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
key = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
value = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
attention_result = Attention(use_scale=True)([query, key, value])
combined = Add()([x, attention_result])


# Flatten combined outputs for Dense Layer
x = Flatten()(combined)
x = Dense(100, kernel_regularizer=l2(0.01))(x)
x = LeakyReLU()(x)
x = Dropout(0.5)(x)
output = Dense(4)(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

X_s, y_s = reshape_data(data_s_X, data_s_y)

# Splitting training/testing data
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Model training
model.fit(X_train_s, y_train_s, epochs=100, batch_size=32, validation_data=(X_test_s, y_test_s), callbacks=[es])

# Predictions
predictions_s = model.predict(X_test_s)

# Print predictions
print("Predictions for data_s")
print(predictions_s[:5])

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 10, 29)]     0           []                               
                                                                                                  
 bidirectional_5 (Bidirectional  (None, 10, 200)     104000      ['input_2[0][0]']                
 )                                                                                                
                                                                                                  
 layer_normalization_2 (LayerNo  (None, 10, 200)     400         ['bidirectional_5[0][0]']        
 rmalization)                                                                                     
                                                                                            

In [14]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])
predictions_40s.to_csv('/content/drive/My Drive/철도/40s_20230821_lstm.csv', index=False)

In [15]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2", "A_M1_B1_W1_Z_L", "A_M1_B1_BC_Z_L", "A_M1_B1_W2_Z_L", "A_M1_B1_W1_Z_R", "A_M1_B1_BC_Z_R", "A_M1_B1_W2_Z_R","Gauge"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # Save column names before transformation
    columns = X.columns.tolist()

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, columns

def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y, column_names = prepare_data(s50_train)

# Model building
N_TIMESTEPS = 10
N_FEATURES = 29

input_layer = Input(shape=(N_TIMESTEPS, N_FEATURES))
x = Bidirectional(LSTM(100, return_sequences=True))(input_layer)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(100, return_sequences=True))(x)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

query = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
key = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
value = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
attention_result = Attention(use_scale=True)([query, key, value])
combined = Add()([x, attention_result])


# Flatten combined outputs for Dense Layer
x = Flatten()(combined)
x = Dense(100, kernel_regularizer=l2(0.01))(x)
x = LeakyReLU()(x)
x = Dropout(0.5)(x)
output = Dense(4)(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

X_s, y_s = reshape_data(data_s_X, data_s_y)

# Splitting training/testing data
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Model training
model.fit(X_train_s, y_train_s, epochs=100, batch_size=32, validation_data=(X_test_s, y_test_s), callbacks=[es])

# Predictions
predictions_s = model.predict(X_test_s)

# Print predictions
print("Predictions for data_s")
print(predictions_s[:5])

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 10, 29)]     0           []                               
                                                                                                  
 bidirectional_10 (Bidirectiona  (None, 10, 200)     104000      ['input_3[0][0]']                
 l)                                                                                               
                                                                                                  
 layer_normalization_4 (LayerNo  (None, 10, 200)     400         ['bidirectional_10[0][0]']       
 rmalization)                                                                                     
                                                                                            

In [ ]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])
predictions_50s.to_csv('/content/drive/My Drive/철도/50s_20230821_lstm.csv', index=False)

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2", "A_M1_B1_W1_Z_L", "A_M1_B1_BC_Z_L", "A_M1_B1_W2_Z_L", "A_M1_B1_W1_Z_R", "A_M1_B1_BC_Z_R", "A_M1_B1_W2_Z_R","Gauge"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # Save column names before transformation
    columns = X.columns.tolist()

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, columns

def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y, column_names = prepare_data(s70_train)

# Model building
N_TIMESTEPS = 10
N_FEATURES = 29

input_layer = Input(shape=(N_TIMESTEPS, N_FEATURES))
x = Bidirectional(LSTM(100, return_sequences=True))(input_layer)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(100, return_sequences=True))(x)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

query = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
key = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
value = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
attention_result = Attention(use_scale=True)([query, key, value])
combined = Add()([x, attention_result])


# Flatten combined outputs for Dense Layer
x = Flatten()(combined)
x = Dense(100, kernel_regularizer=l2(0.01))(x)
x = LeakyReLU()(x)
x = Dropout(0.5)(x)
output = Dense(4)(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

X_s, y_s = reshape_data(data_s_X, data_s_y)

# Splitting training/testing data
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Model training
model.fit(X_train_s, y_train_s, epochs=100, batch_size=32, validation_data=(X_test_s, y_test_s), callbacks=[es])

# Predictions
predictions_s = model.predict(X_test_s)

# Print predictions
print("Predictions for data_s")
print(predictions_s[:5])

In [ ]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])
predictions_70s.to_csv('/content/drive/My Drive/철도/70s_20230821_lstm.csv', index=False)

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2", "A_M1_B1_W1_Z_L", "A_M1_B1_BC_Z_L", "A_M1_B1_W2_Z_L", "A_M1_B1_W1_Z_R", "A_M1_B1_BC_Z_R", "A_M1_B1_W2_Z_R","Gauge"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # Save column names before transformation
    columns = X.columns.tolist()

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, columns

def reshape_data(X, y):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y, column_names = prepare_data(s100_train)

# Model building
N_TIMESTEPS = 10
N_FEATURES = 29

input_layer = Input(shape=(N_TIMESTEPS, N_FEATURES))
x = Bidirectional(LSTM(100, return_sequences=True))(input_layer)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(100, return_sequences=True))(x)
x = LayerNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.3)(x)

query = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
key = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
value = Bidirectional(LSTM(100, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01)))(x)
attention_result = Attention(use_scale=True)([query, key, value])
combined = Add()([x, attention_result])


# Flatten combined outputs for Dense Layer
x = Flatten()(combined)
x = Dense(100, kernel_regularizer=l2(0.01))(x)
x = LeakyReLU()(x)
x = Dropout(0.5)(x)
output = Dense(4)(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

X_s, y_s = reshape_data(data_s_X, data_s_y)

# Splitting training/testing data
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Model training
model.fit(X_train_s, y_train_s, epochs=100, batch_size=32, validation_data=(X_test_s, y_test_s), callbacks=[es])

# Predictions
predictions_s = model.predict(X_test_s)

# Print predictions
print("Predictions for data_s")
print(predictions_s[:5])

In [ ]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])
predictions_100s.to_csv('/content/drive/My Drive/철도/100s_20230821_lstm.csv', index=False)

In [ ]:
# # 예측 결과 DataFrame 생성
# predictions_df = pd.DataFrame(predictions_s, columns=["Pred_YL_M1_B1_W1", "Pred_YR_M1_B1_W1", "Pred_YL_M1_B1_W2", "Pred_YR_M1_B1_W2"])

# # CSV로 저장
# predictions_df.to_csv("/Users/leeyoungdong/predictions100.csv", index=False)